# R1 final ensemble runs — historic 61/31/46 split

**Purpose:** This notebook applies **one manually entered, development-selected ensemble configuration** to the completed `metrics2026.xlsx` workbook. It creates 100 run-matched ensemble runs and writes one simple raw Excel workbook.

> **Boundary:** This notebook does not rank architectures, rank ensemble configurations, calculate median summaries, calculate confidence intervals, or choose a test-set winner. Later paper reporting will combine this raw 100-run EL workbook with the 1,100 TL runs, 100 CNN runs, and the clinical-reader output.

For seed 1, it takes seed-1 predictions from every selected constituent model and applies the selected rule. It repeats this for seeds 2–100. No model is retrained and no additional patients are used.


## 0. Secure paths and manually entered configuration

Copy the Top-3/Top-5 lists and selected pool/rule from the middle development-only notebook. The Top-3 list must be the first three entries of Top-5. The selected rule must be one of `sum`, `median`, `max`, `majority_vote`, or `weighted_sum`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Run from the repository root, `notebooks/`, or `Experiment2026/`.
CWD = Path.cwd().resolve()
if (CWD / 'analysis').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'analysis').exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError('Open this notebook from the repository root, notebooks/, or Experiment2026/.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.r1_ensemble_core import (
    create_run_matched_ensemble_predictions,
    load_historic_tl_workbook,
    raw_ensemble_run_results,
    raw_ensemble_study_lock,
    validate_manual_configuration,
)

# Secure local paths. These files must never be committed to Git.
TL_WORKBOOK = REPO_ROOT / 'Experiment2026' / 'metrics2026.xlsx'
ANALYSIS_OUTPUT_DIR = REPO_ROOT.parent / 'secure_analysis' / 'R1_final_ensemble_2026'
OUTPUT_WORKBOOK = ANALYSIS_OUTPUT_DIR / 'R1_final_ensemble_raw_runs.xlsx'

# Enter these names manually after reviewing the middle development-only notebook.
TOP3_MODELS = [
    # 'Architecture_1',
    # 'Architecture_2',
    # 'Architecture_3',
]
TOP5_MODELS = [
    # 'Architecture_1',
    # 'Architecture_2',
    # 'Architecture_3',
    # 'Architecture_4',
    # 'Architecture_5',
]
SELECTED_POOL = 'Top-5'      # 'Top-3' or 'Top-5'
SELECTED_RULE = 'max'        # 'sum', 'median', 'max', 'majority_vote', or 'weighted_sum'
# Only for SELECTED_RULE='weighted_sum': enter the development-only weights.
WEIGHTED_SUM_WEIGHTS = {}

# Keep False until the manually entered configuration above is complete.
RUN_ANALYSIS = False

EXPECTED_MODELS = (
    'VGG16', 'VGG19', 'ResNet50', 'ResNet101', 'ResNet152', 'InceptionV3',
    'InceptionResNetV2', 'DenseNet169', 'DenseNet201', 'MobileNetV2', 'Xception',
)

print('Historic 1,100-run workbook:', TL_WORKBOOK)
print('Raw ensemble output workbook:', OUTPUT_WORKBOOK)
print('Selected pool/rule:', SELECTED_POOL, '/', SELECTED_RULE)
print('This notebook runs one manually entered configuration only.')


## 1. Validate the manually entered configuration

This checks the pool/rule structure only. It does not read test predictions.


In [ ]:
configuration = validate_manual_configuration(
    top3_models=TOP3_MODELS,
    top5_models=TOP5_MODELS,
    selected_pool=SELECTED_POOL,
    selected_rule=SELECTED_RULE,
    weighted_sum_weights=WEIGHTED_SUM_WEIGHTS,
)
display(pd.DataFrame([configuration]))


## 2. Read the completed 1,100-run TL workbook

The input must contain exactly 11 architectures × 100 seeds and 46 ordered test predictions per model/seed. It is opened read-only.


In [ ]:
if not RUN_ANALYSIS:
    print('Set RUN_ANALYSIS = True only after confirming the manual configuration above.')
else:
    historic_predictions, test_manifest, study_lock = load_historic_tl_workbook(TL_WORKBOOK, EXPECTED_MODELS)
    print(f'Validated {len(historic_predictions)} prediction records.')
    print(f'Architectures: {historic_predictions.model_name.nunique()} | seeds: {historic_predictions.seed.nunique()} | test patients: {historic_predictions.patient_id.nunique()}')


## 3. Create the 100 run-matched ensemble runs

Each ensemble seed uses only same-seed constituent predictions. This cell creates raw prediction and metric records for the 100 ensemble runs; it does not calculate any paper summaries.


In [ ]:
if RUN_ANALYSIS:
    ensemble_predictions, ensemble_run_metrics = create_run_matched_ensemble_predictions(
        historic_predictions=historic_predictions,
        configuration=configuration,
    )
    assert len(ensemble_predictions) == 100 * 46
    assert len(ensemble_run_metrics) == 100
    display(ensemble_run_metrics.head())
    print('Created 100 run-matched ensemble runs with 46 predictions per run.')


## 4. Write the raw EL workbook

The workbook is intentionally simple. It contains only the raw ensemble runs, their ordered 46-patient probabilities, the test manifest, and a short provenance sheet. A later combined-analysis notebook will merge this with TL, CNN, and clinical-reader inputs for medians, patient-level 95% CIs, tables, and figures.


In [ ]:
if RUN_ANALYSIS:
    ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    raw_run_results = raw_ensemble_run_results(
        ensemble_predictions=ensemble_predictions,
        ensemble_run_metrics=ensemble_run_metrics,
        configuration=configuration,
    )
    raw_lock = raw_ensemble_study_lock(configuration, TL_WORKBOOK)
    with pd.ExcelWriter(OUTPUT_WORKBOOK, engine='openpyxl', mode='w') as writer:
        raw_run_results.to_excel(writer, sheet_name='Run_results', index=False)
        test_manifest.to_excel(writer, sheet_name='Patient_manifest', index=False)
        raw_lock.to_excel(writer, sheet_name='Study_lock', index=False)
    print('Wrote raw 100-run ensemble workbook:', OUTPUT_WORKBOOK)
    print('Sheets: Run_results | Patient_manifest | Study_lock')


## Raw output

```text
secure_analysis/R1_final_ensemble_2026/
└── R1_final_ensemble_raw_runs.xlsx
    ├── Run_results       # 100 raw ensemble runs with metrics and 46 ordered probabilities
    ├── Patient_manifest  # test-patient order and labels
    └── Study_lock        # manually entered pool/rule and seed-matching provenance
```

This is the EL input for the later combined reporting notebook. It will be combined with the original 1,100 TL runs, 100 CNN runs, and the clinical reader—not used to select any further ensemble configuration.
